# Payroll Data Masking & Privacy-Preserving Data Preparation

This Jupyter Notebook documents the data preparation and masking process for a proof-of-concept Power BI payroll analytics dashboard, developed as part of a proposed business intelligence solution for executive payroll reporting and workforce analytics. 

The notebook transforms the source payroll dataset into a sanitized and masked version by removing sensitive personal identifiers and applying masking techniques to employee information, organizational data, job designations, and payroll values while maintaining the underlying relationships and analytical structure of the dataset. This process allows the resulting data to retain its usefulness for demonstrating payroll KPIs, trends, workforce metrics, and interactive Power BI reporting without exposing the original sensitive information. 

The masked dataset is intended strictly for portfolio, demonstration, and proof-of-concept purposes, providing a safe representation of the proposed analytics solution without displaying confidential or personally identifiable payroll data.

In [ ]:
# Pandas is used for loading, manipulating, validating, and exporting tabular data.
import pandas as pd
# NumPy is used for numerical calculations and generating random values
# for the masking process.
import numpy as np  

# Faker is used to generate synthetic employee names so the original
# employee names are not exposed.
from faker import Faker
fake = Faker()

# This is used to measure how long the Excel export process takes.
import time


This code is necessary for data loading and transformation in order to anonymize and masked sensitive data.

## Data Importing

In [ ]:
data_source = r"source.xlsx"

In [ ]:
'''
`pd.read_excel(...)`: reads Excel data into Pandas.
`sheet_name=None`: tells Pandas to load **all worksheets**, returning them as a dictionary of DataFrames.
'''
df = pd.read_excel(data_source, sheet_name=None)

### Cost Centers

Masks potentially sensitive organizational names while maintaining consistent replacement values. Using a mapping means the same source value receives the same masked value wherever it appears.

In [ ]:
# Create a mapping from the original cost center names to masked names,
# then replace the original values using the mapping.
df["costcenter"]["CCN_NAME"] = df["costcenter"]["CCN_NAME"].replace(ccn_mapping)

# Display the unique masked cost center names for verification.
df["costcenter"]["CCN_NAME"].unique()  

array(['Finance', 'Operations', 'IT Services', 'Maintenance',
       'Production', 'Assembly', 'Processing', 'Finishing', 'Cutting',
       'Polishing', 'Production Office', 'Quality Assurance', 'Warehouse',
       'Executive', 'Specialist', 'Administration'], dtype=object)

The output displays the unique cost center names present after the masking process. The resulting dataset contains 16 distinct cost centers, covering functions such as Finance, Operations, IT Services, Production, Administration, and Executive.

The output demonstrates that the cost center field remains populated with meaningful categorical values after transformation. The organizational structure is therefore retained for analytical purposes while the original identifiers can be replaced with masked equivalents.

In [ ]:
# Create a mapping from the original cost center codes to masked codes,
# then replace the original values using the mapping.
df["costcenter"]["CCN_CODE"] = df["costcenter"]["CCN_CODE"].replace(ccncode_mapping)

# Display the unique masked cost center code for verification.
df["costcenter"]["CCN_CODE"].unique()

array(['FIN', 'OPS', 'IT', 'MNT', 'PRD', 'ASY', 'PRC', 'FSH', 'CUT',
       'POL', 'POF', 'QA', 'WHS', 'EXE', 'SPC', 'ADM'], dtype=object)

The output confirms that the cost center codes have been transformed into standardized masked codes. Sixteen unique codes are present, corresponding to the available cost center categories.

### Depm

In [ ]:
# Create a mapping from the original department names to department names,
# then replace the original values using the mapping.
df["depm"]["DEP_NAME"] = df["depm"]["DEP_NAME"].replace(dep_mapping)

# Display the unique masked department names for verification.
df["depm"]["DEP_NAME"].unique()

array(['Administration', 'Production', 'Cutting', 'Processing',
       'Polishing', 'Grinding', 'Coating', 'Assembly', 'Packaging',
       'Warehouse', 'Maintenance', 'Production Office',
       'Quality Assurance'], dtype=object)

The output shows the unique department categories available after the transformation. The department information remains categorical and usable for workforce segmentation and payroll analysis.

Maintaining these categories is important because removing or randomizing organizational classifications would reduce the usefulness of the dataset for analytical demonstrations.

### Group ID

In [13]:
# Create a mapping from the original group names to group names,
# then replace the original values using the mapping.
df["groupid"]["NAME"] = df["groupid"]["NAME"].replace(group_mapping)

# Display the unique masked group names for verification.
df["groupid"]["NAME"].unique()

array(['Human Resources', 'Operations', 'Packaging', 'Grinding'],
      dtype=object)

In [ ]:
# Create a mapping from the original group codes to group codes,
# then replace the original values using the mapping.
df["groupid"]["CODE"] = df["groupid"]["CODE"].replace(gripcode_mapping)

# Display the unique masked group codes for verification.
df["groupid"]["CODE"].unique()

array(['HR', 'OPS', 'PKG', 'GRD'], dtype=object)

In [ ]:
# Print the entire groupid DataFrame.
# This allows the complete transformed group table to be inspected.
print(df["groupid"])

  CODE             NAME
0   HR  Human Resources
1  OPS       Operations
2  PKG        Packaging
3  GRD         Grinding


The resulting groupid table contains four group records. Each record has both a masked group code and its corresponding group name.

This demonstrates that the masking process preserved the relationship between the code and its descriptive category. The table can therefore continue to function as a lookup/reference table for analytical relationships.

### payhisto

In [ ]:
# Create a list containing sensitive columns that should not be included
# in the masked dataset.
remove_cols = [
    # Employee bank account information.
    "BANKACCT",

    # Employee Tax Identification Number.
    "EMPL_TIN",

    # Employee SSS number.
    "EMSSSNUM",

    # Employee Pag-IBIG number.
    "PAGIBIG_NO",

    # Employee PhilHealth number.
    "PHIC_NO"
]

# Permanently remove the sensitive columns from the payhisto DataFrame.
df["payhisto"].drop(columns=remove_cols, inplace=True)

In [16]:
df["payhisto"]["COST_CENTE"] = df["payhisto"]["COST_CENTE"].replace(ccncode_mapping)

costcenter_values = set(df["costcenter"]["CCN_CODE"].dropna().unique())
payhisto_values = set(df["payhisto"]["COST_CENTE"].dropna().unique())

print("Only in costcenter:")
print(costcenter_values - payhisto_values)

print("\nOnly in payhisto:")
print(payhisto_values - costcenter_values)

print("\nAre they exactly the same?")
print(costcenter_values == payhisto_values)

Only in costcenter:
set()

Only in payhisto:
set()

Are they exactly the same?
True


The validation confirms that the unique cost center codes in the costcenter table and the payhisto table are identical after masking.

In [17]:
df["payhisto"]["IDENTI"] = df["payhisto"]["IDENTI"].replace(gripcode_mapping)

groupid_values = set(df["groupid"]["CODE"].dropna().unique())
payhisto_values = set(df["payhisto"]["IDENTI"].dropna().unique())

print("Only in GroupID:")
print(groupid_values - payhisto_values)

print("\nOnly in payhisto:")
print(payhisto_values - groupid_values)

print("\nAre they exactly the same?")
print(groupid_values == payhisto_values)

Only in GroupID:
set()

Only in payhisto:
set()

Are they exactly the same?
True


The output confirms that the group identifiers used by the groupid table and the payhisto table remain consistent after masking.

In [ ]:
df["payhisto"]["DESIGNATIO"] = df["payhisto"]["DESIGNATIO"].replace(role_map)
df["payhisto"]["DESIGNATIO"].unique()

array(['Production Staff', 'Production Department', 'Team Lead',
       'Admin Team Lead', 'IT Team Lead', 'Support Staff',
       'Logistics Staff', 'Executive', 'Supervisor', 'Operations Manager',
       'Production Supervisor', 'Consultant', 'HR/Admin Staff',
       'IT Staff', 'General Manager', 'Finance Staff'], dtype=object)

The output shows the unique job designations remaining after the masking process. Sixteen distinct designation categories are represented, ranging from production and support positions to managerial and executive roles.

# Money related masking

In [ ]:
#  Create a list containing all payroll-related monetary columns
#  that will be transformed/masked later.
money_cols = [
    "RATE", # Employee rate or salary rate.
    "GROSS_YEAR",   # Annual gross salary.
    "YEARLY_TAX",   # Annual tax amount.
    "NET_YEAR", # Annual net salary.
    "NET_DUE_YE",   # Annual net amount due.
    "AMT_13MO", # 13th-month pay amount.
    "AMTBONUS", # Bonus amount.
    "ALLOWMPT", # Allowance amount.
    "AMT__REG", # Regular pay amount.
    "AMTHRATE", # Hourly-rate-related amount.
    "AMTMRATE", # Monthly-rate-related amount.
    "AMTUTIME", # Undertime amount.
    "SAL__DED", # Salary deduction.
    "AMT___ND", # Night differential amount.
    "AMT___OT", # Overtime amount.
    "AMT_EXSS", # Excess SSS-related amount.
    "AMT_OTND", # Overtime/night-differential amount.
    "AMT_ADJ1", # First adjustment amount.
    "AMT_ADJ2", # Second adjustment amount.
    "AMT13MO1", # 13th-month-related amount.
    "BONUS",    # Bonus amount.
    "AMTBONU1", # Additional bonus amount.
    "ALLOWR",   # Regular allowance.
    "AMTEXMPT", # Exempt amount.
    "AMTGROSS", # Gross amount.
    "SSS_EMPL", # Employee SSS contribution.
    "SSS_EMPR", # Employer SSS contribution.
    "PROVEMPL", # Employee provident-fund contribution.
    "PROVEMPR", # Employer provident-fund contribution.
    "UNIONDUES",    # Union dues.
    "PBG_EMPL", # Employee Pag-IBIG contribution.
    "PBG_EMPR", # Employer Pag-IBIG contribution.
    "MED_EMPL", # Employee medical/health contribution.
    "MED_EMPR", # Employer medical/health contribution.
    "AMT_WTAX", # Withholding tax amount.
    "AMT__DUE", # Amount due.
    "AMT__NET", # Net amount.
    "TAXABLE"   # Taxable amount.
]

In [ ]:
# Define a function that receives a complete name
# and separates it into last name, first name, and middle name.
def split_name(full_name):

    # Split the full name wherever there is whitespace.
    # Example:
    # "John Michael Smith"
    # becomes ["John", "Michael", "Smith"].
    parts = full_name.split()

    # The first element is treated as the first name.
    fname = parts[0]

    # The last element is treated as the last name.
    lname = parts[-1]

    # If there are more than two name parts, use the second part
    # as the middle name.
    # Otherwise, use an empty string.
    mname = parts[1] if len(parts) > 2 else ""

    # Return the values as a pandas Series.
    # The order is:
    # last name, first name, middle name.
    return pd.Series([lname, fname, mname])

In [ ]:
# Set NumPy's random seed to 42.
# This makes the random masking values reproducible when the code
# is executed again using the same data and environment.
np.random.seed(42)

# --------------------------------
# 1. Get original employee numbers
# --------------------------------
# Get all unique employee numbers from payhisto.
unique_ids = df["payhisto"]["EMP_NUMB"].dropna().unique()

# --------------------------------
# 2. Create masked employee IDs
# --------------------------------
# Create a dictionary that maps each original employee ID
# to a new synthetic employee ID.
id_map = {
    emp_id: f"EMP-{1000+i}"
    for i, emp_id in enumerate(unique_ids)
}

# Replace every original employee ID with its corresponding
# masked employee ID using the mapping created above.
df["payhisto"]["EMP_NUMB"] = df["payhisto"]["EMP_NUMB"].map(id_map)

# Retrieve the unique masked employee IDs.
# This replaces the previous unique_ids variable with the new
# masked IDs because they are needed for the name-generation step.
unique_ids = df["payhisto"]["EMP_NUMB"].dropna().unique()

# --------------------------------
# 3. Create masked employee names
# --------------------------------
# Create a dictionary that assigns one fake name to every
# masked employee ID.
#
# fake.name() generates a synthetic name such as:
# "John Smith" or "Maria Johnson".
name_map = {
    emp_id: fake.name()
    for emp_id in unique_ids
}

# Add the generated fake name to a temporary column.
# The employee ID is used to look up the correct fake name.
df["payhisto"]["FULL_FAKE_NAME"] = df["payhisto"]["EMP_NUMB"].map(name_map)

# The split_name function created earlier performs this operation.
# The resulting three values are assigned back to the original
# employee-name columns.
df["payhisto"][["EMPLNAME", "EMPFNAME", "EMPMNAME"]] = df["payhisto"]["FULL_FAKE_NAME"].apply(split_name)
df["payhisto"].drop(columns=["FULL_FAKE_NAME"], inplace=True)

# --------------------------------
# 4. Create employee-specific
#    salary multipliers
# --------------------------------
# Create a different random multiplier for each employee.
#
# np.random.uniform(1.55, 1.90) generates a random number
# between 1.55 and 1.90.
#
# Each employee therefore gets their own salary-scaling factor.
employee_multiplier = {
    masked_id: np.random.uniform(1.55, 1.90)
    for masked_id in df["payhisto"]["EMP_NUMB"].dropna().unique()
}

# --------------------------------
# 5. Transform monetary columns
# --------------------------------
# Process every monetary column listed in money_cols.
for col in money_cols:

    # Each employee gets their own multiplier
    multiplier = df["payhisto"]["EMP_NUMB"].map(employee_multiplier)

    # Generate a small random variation for every payroll record.
    # This prevents every payroll record from being transformed
    # by exactly the same factor.
    noise = np.random.uniform(
        0.92,
        1.08,
        len(df["payhisto"])
    )

    # Apply the masking transformation:
    #
    # original value
    #     × employee multiplier
    #     × record-level random noise
    #
    # round(2) keeps the resulting monetary value at two decimal places.
    df["payhisto"][col] = (
        df["payhisto"][col]
        * multiplier
        * noise
    ).round(2)

# Display the resulting payhisto DataFrame
# so the transformed dataset can be inspected.
df["payhisto"]

,EMP_NUMB,EXP_3,EMPLNAME,EMPFNAME,EMPMNAME,REC_TYPE,CIVLSTAT,EMPCLASS,EMPL_SEX,MASEXEMPT,...,PROVEMPR,UNIONDUES,PBG_EMPL,PBG_EMPR,MED_EMPL,MED_EMPR,AMT_WTAX,AMT__DUE,AMT__NET,TAXABLE
0,EMP-1000,False,Brown,Katherine,,DEFAULT,M,R,M,0,...,NaN,0.0,0.00,0.00,0.00,0.00,0.00,12417.09,16377.92,13789.43
1,EMP-1001,False,Reilly,Matthew,,DEFAULT,M,R,F,0,...,NaN,0.0,0.00,0.00,0.00,0.00,0.00,9982.80,18644.50,14068.42
2,EMP-1002,False,Conway,Johnathan,,DEFAULT,M,R,M,0,...,NaN,0.0,0.00,0.00,0.00,0.00,0.00,9211.31,13745.28,11936.64
3,EMP-1003,False,Ortiz,Daniel,,DEFAULT,S,R,F,0,...,NaN,0.0,0.00,0.00,0.00,0.00,0.00,10315.97,14613.88,12585.56
4,EMP-1004,False,Freeman,Jocelyn,,DEFAULT,S,R,F,0,...,NaN,0.0,0.00,0.00,0.00,0.00,0.00,15673.11,17478.19,15064.37
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7546,EMP-1071,False,Martin,Kimberly,,DEFAULT,S,R,M,0,...,0.0,0.0,0.00,0.00,429.75,461.01,0.00,-1546.57,-462.44,-413.54
7547,EMP-1072,False,Parker,Kelsey,,DEFAULT,S,R,F,0,...,0.0,0.0,0.00,0.00,438.75,385.46,0.00,-4974.07,-423.13,-388.83
7548,EMP-1073,False,Jenkins,Maria,,DEFAULT,M,R,M,0,...,0.0,0.0,0.00,0.00,517.99,450.14,0.00,-2580.80,-470.01,-501.60
7549,EMP-1074,False,Anderson,Sandra,,DEFAULT,M,R,F,0,...,0.0,0.0,0.00,0.00,460.22,460.63,0.00,-497.74,-510.02,-485.17


The resulting payhisto DataFrame demonstrates the completed privacy-preserving transformation of the payroll dataset. Employee identifiers and names have been replaced with synthetic values, while sensitive personal identifiers such as bank account, TIN, SSS, Pag-IBIG, and PhilHealth information have been removed.

The output retains the fields necessary for payroll and workforce analytics, including employee classifications, departments, designations, cost centers, payroll periods, attendance information, compensation, deductions, employer/employee contributions, taxes, and net payroll values.

Therefore, the output demonstrates that the dataset has been sanitized while retaining sufficient analytical structure for use in a proof-of-concept payroll dashboard or business-intelligence solution.

In [ ]:
# Display information about the payhisto DataFrame.
df["payhisto"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7551 entries, 0 to 7550
Data columns (total 64 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   EMP_NUMB    7551 non-null   object 
 1   EXP_3       7551 non-null   bool   
 2   EMPLNAME    7551 non-null   object 
 3   EMPFNAME    7551 non-null   object 
 4   EMPMNAME    7551 non-null   object 
 5   REC_TYPE    7551 non-null   object 
 6   CIVLSTAT    7551 non-null   object 
 7   EMPCLASS    7312 non-null   object 
 8   EMPL_SEX    7551 non-null   object 
 9   MASEXEMPT   7551 non-null   int64  
 10  DEPARTMENT  7551 non-null   object 
 11  IDENTI      1444 non-null   object 
 12  RATE_TYPE   7551 non-null   object 
 13  RATE        7551 non-null   float64
 14  GROSS_YEAR  7551 non-null   float64
 15  RANK_CODE   7551 non-null   int64  
 16  DESIGNATIO  7551 non-null   object 
 17  COST_CENTE  7551 non-null   object 
 18  CUSTOM1     0 non-null      float64
 19  CUSTOM2     0 non-null     

In [ ]:
output_path = r"output.xlsx"

# Define the worksheets that should be included in the output workbook.
sheets_to_export = [
    "costcenter",
    "depm",
    "empClass",
    "groupid",
    "payhisto"
]

start_time = time.time()

# Create an Excel writer using the openpyxl engine.
# The "with" statement automatically closes/saves the Excel writer
# when the block finishes.
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for i, sheet_name in enumerate(sheets_to_export, start=1):
        
        sheet_start = time.time()
        print(f"[{i}/{len(sheets_to_export)}] Writing sheet: {sheet_name} ...")

        sheet_df = df[sheet_name]

        # Write the DataFrame to the Excel workbook.
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)

        sheet_time = time.time() - sheet_start
        print(f"✔ Done: {sheet_name} in {sheet_time:.2f} seconds")

total_time = time.time() - start_time


print(
    f"\nAll {len(sheets_to_export)} sheets "
    f"successfully written in {total_time:.2f} seconds!"
)

[1/5] Writing sheet: costcenter ...
✔ Done: costcenter in 0.03 seconds
[2/5] Writing sheet: depm ...
✔ Done: depm in 0.01 seconds
[3/5] Writing sheet: empClass ...
✔ Done: empClass in 0.00 seconds
[4/5] Writing sheet: groupid ...
✔ Done: groupid in 0.00 seconds
[5/5] Writing sheet: payhisto ...
✔ Done: payhisto in 6.66 seconds

All 5 sheets successfully written in 19.79 seconds!
